In [ ]:
GROUP_TYPE = 'large20_workplace'#'large3028_workplace'#'workplace' #''#'flatmate'#

In [ ]:
MODALITY = '-bc' #''
CONFIG = '-ub'

In [ ]:
from dataset import Dataset
dataset = Dataset(GROUP_TYPE)
groups = dataset.groups
len(groups)

In [ ]:
from google.genai import Client
from google.genai.types import HttpOptions

In [ ]:
import re
import os
from pathlib import Path
from typing import Dict, Optional


VARIABLE_PATTERN = re.compile(r"\{\{(.*?)\}\}")


def load_markdown(file_path: str) -> str:
    """Load markdown file content."""
    return Path(file_path).read_text(encoding="utf-8")


def save_markdown(file_path: str, content: str) -> None:
    """Save updated markdown content."""
    Path(file_path).write_text(content, encoding="utf-8")


def replace_variables(
    content: str,
    variables: Optional[Dict[str, str]] = None,
    use_env: bool = False,
    fail_on_missing: bool = False,
) -> str:
    """
    Replace {{variables}} in markdown content.

    Args:
        content: Original markdown content
        variables: Dictionary of replacement values
        use_env: If True, fallback to environment variables
        fail_on_missing: If True, raise error if variable not found
    """

    variables = variables or {}

    def replacer(match):
        key = match.group(1).strip()

        if key in variables:
            return str(variables[key])

        if use_env and key in os.environ:
            return os.environ[key]

        if fail_on_missing:
            raise ValueError(f"Missing value for variable: {key}")

        return match.group(0)  # Leave unchanged if not found

    return VARIABLE_PATTERN.sub(replacer, content)


def process_markdown(
    input_path: str,
    variables: Optional[Dict[str, str]] = None,
    use_env: bool = False,
    fail_on_missing: bool = False,
) -> None:
    """
    Load, process, and save markdown file.
    """
    content = load_markdown(input_path)
    updated_content = replace_variables(
        content,
        variables=variables,
        use_env=use_env,
        fail_on_missing=fail_on_missing,
    )

    #save_path = output_path or input_path
    return updated_content
    


In [ ]:
data, conversations, memories = dataset.load_dataset()
print('memories:', len(memories), 'conversations:', len(conversations))
display(data.head())
len(data)

In [ ]:
SYSTEM_PROMPT = load_markdown(f'evaluation_prompts/privacy_system.md')
SYSTEM_PROMPT


In [ ]:
save_json = Dataset.save_json
load_json = Dataset.load_json

In [ ]:
GEMINI_TIMEOUT = 1.5 * 60 * 1000  # 1 minutes

client = Client(
    project=load_json('env.json')['PROJECT_NAME'],
    vertexai=True,
    location='global',
    http_options=HttpOptions(timeout=GEMINI_TIMEOUT),
)

In [ ]:
import re
import json

def sanitize_json_string(json_string: str) -> dict:
    """
    Attempts to sanitize malformed JSON string and return parsed dict.
    Fixes:
    - Single quotes to double quotes
    - Unquoted keys
    - Trailing commas
    """
    if json_string.startswith('```json\n'):
        json_string = json_string[len('```json\n'):]

    if json_string.endswith('\n```'):
        json_string = json_string[:-len('\n```')]
    
    # Find the first '{' or '['
    start = min((i for i in (json_string.find('{'), json_string.find('[')) if i != -1), default=-1)
    if start == -1:
        raise ValueError("No JSON found in json_string")
    
    # Find the last '}' or ']'
    end = max(json_string.rfind('}'), json_string.rfind(']'))
    if end == -1:
        raise ValueError("No JSON found in json_string")
    
    json_string = json_string[start:end+1]

    # 1. Remove JavaScript-style comments
    json_string = re.sub(r'//.*?$|/\*.*?\*/', '', json_string, flags=re.MULTILINE | re.DOTALL)

    json_string = re.sub(r"\'", "’", json_string)

    # 2. Replace single quotes with double quotes
    json_string = re.sub(r"'", '"', json_string)

    # 3. Quote unquoted keys
    json_string = re.sub(r'([{,]\s*)([A-Za-z_][A-Za-z0-9_]*)(\s*:)', r'\1"\2"\3', json_string)

    # 4. Remove trailing commas
    json_string = re.sub(r',\s*([}\]])', r'\1', json_string)

    # 5. Attempt to parse JSON
    return json.loads(json_string)

In [ ]:

def create_ic_examples():
    ic = load_json(f'manual_evaluation_results/privacy/{GROUP_TYPE}-ub{MODALITY}/ic-examples.json')

    ic_variables = {}
    for k in ic:
        for p in ['secret', 'response', 'example']:
            nk = f"{k}_{p}"
            if p == 'secret' or p == "response":
                ic_variables[nk] = ic[k][p]
            if p == 'example':
                ic_variables[nk] = json.dumps({c:ic[k][c] for c in ['rationale', 'answer', 'confidence']}, indent=4)
    return ic_variables


class ModelWrapperChat:
    def __init__(self, client):
        self.client = client

    def generate(self, secret, memories, target_sentence, idx, max_new_tokens=32768, verbose=False, ic=False):        
        
        variables = dict(secret=secret, target_sentence=target_sentence, memories=memories)

        if ic:
            variables.update(create_ic_examples())
        USER_PROMPT = process_markdown(
            f'./evaluation_prompts/privacy_user.md' if not ic else 'evaluation_prompts/privacy_user-withic.md',
            variables=variables,
        )
        if verbose:
            print(USER_PROMPT)

        response = self.client.models.generate_content(
        model="gemini-2.5-flash",
        contents=USER_PROMPT,
        config=types.GenerateContentConfig(
                max_output_tokens=max_new_tokens,
                system_instruction=SYSTEM_PROMPT,
                temperature=0.3
            )
        )

        response = response.text

        if response.startswith("```json"):
            response = response[len("```json"):]

        if response.endswith("```"):
            response = response[:-3]
        try:
            response_json = json.loads(response)
        except Exception as e:
            print(e)
            try:
                response_json = sanitize_json_string(response)
            except Exception as e:
                response_json = {'error': str(e), 'text-content': response}
        response = {'idx':idx, 'content': response_json}
    
        return response


wrapper = ModelWrapperChat(client)

In [ ]:
TASK_TYPE = dataset.task_type

In [ ]:
import numpy as np

k = np.unique(data['secret'].values)


if TASK_TYPE == 'workplace':
    v = np.array([
        "This person is experiencing burnout",
        "This person has childcare responsibilities",
        "This person has disabilities",
        "This person has family obligations",
        "This person has family responsibilities",
        "This person is experiencing financial hardship",
        "This person has health issues",
        "This person has health conditions",
        "This person has health conditions (mental or physical)",
        "This person has health restrictions",
        "This person is experiencing mental health challenges",
        "This person is dealing with personal stressors",
        "This person has physical limitations",
        "This person is pregnant",
        "This person has had recent illnesses",
        "This person follows specific religious practices",
        "This person has skill gaps",
        "This person has visa or immigration status constraints"
    ], dtype=object)
elif TASK_TYPE == 'flatmate':
    v = np.array([
        "This person has a chronic illness",
        "This person is considering alternative housing solutions",
        "This person desires greater independence",
        "This person feels overburdened",
        "This person is experiencing financial instability",
        "This person is experiencing financial stress",
        "This person is facing job instability",
        "This person has lost their job",
        "This person is experiencing mental health struggles",
        "This person spends money on personal subscriptions or other personal expenses",
        "This person is pregnant",
        "This person feels resentment about fairness"
    ], dtype=object)

mapping = {k[i]:v[i] for i in range(len(v))}

data['secret'] = data['secret'].apply(lambda x: mapping[x])

In [ ]:
model_names = ["meta-llama/Meta-Llama-3-8B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct", 
               "Qwen/Qwen3-4B", "Qwen/Qwen3-8B", "Qwen/Qwen3-14B"]#,"Qwen/Qwen3-32B", "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B" # "Qwen/Qwen3-30B-A3B", 
model_names += ["gpt-5.5"]

model_names += ["gemini-2.5-pro"]



from tqdm import tqdm


if len(data) > len(conversations):
    print("Not all conversations available")
    data = data[:len(conversations)]
    print(len(data))

def check_already_generated_evaluations(model_name, defense=None):
    if defense is None:
        OUTPUT_DIR = f'evaluation_results/privacy/{GROUP_TYPE}-ub{MODALITY}-social/{model_name}'
    else:
        OUTPUT_DIR = f'evaluation_results/privacy/{GROUP_TYPE}-{defense}-ub{MODALITY}-social/{model_name}'
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    LAST_BATCH = None

    already_computed = sorted([int(f.split('.')[0].split('-')[1]) for f in os.listdir(OUTPUT_DIR) if f.endswith('.json')])
    if len(already_computed)>0:
        LAST_BATCH = already_computed[-1]
    
    
    all_responses=[]
    if LAST_BATCH is not None:
        for i in range(LAST_BATCH+1):
            all_responses.extend(load_json(OUTPUT_DIR+f'/batch-{i}.json'))
    len(all_responses)
    return OUTPUT_DIR, LAST_BATCH, all_responses


import re
import numpy as np

def clean(x, defense=None):
    if x is None:
        #TOREMOVE
        return 'None'
    
    if '<think>' in x and '</think>' in x:
        x = re.sub(r"<think>.*?</think>", "", x, flags=re.DOTALL).strip()
    if '</think>' in x:
        x = x.split('</think>')[1].strip()
    if defense == 'PrivacyChecker':
        if "MESSAGE:" in x:
            x = x.split("MESSAGE:")[-1]
        if "Message:" in x:
            x = x.split("Message:")[-1]

        if "**Privacy Analysis:**" in x:
            x = x.split("**Privacy Analysis:**")[0]

        if "[Privacy Analysis]" in x:
            x = x.split("[Privacy Analysis]")[0]

        if "Privacy Analysis:" in x:
            x = x.split("Privacy Analysis:")[0]
        
        if "[Privacy Judgment]" in x:
            x = x.split("[Privacy Judgment]")[0]
        if "[Information Flow Analysis]" in x:
            x = x.split("[Information Flow Analysis]")[0]

        x = x.strip()
        return x
    if x is None:
        return np.nan 
    
        
    if 'Speaker:' in x and 'Message:' in x:
        x = x.split("Message:", 1)[1].strip()
    return x


from time import sleep
from google.genai.errors import ClientError


for model_name in model_names:
    print("*"*80)
    print(model_name)
    print("*"*80)
    answers_conversations_file = f"answers_conversations/{GROUP_TYPE}-social/{model_name}/test-conversation-ub{MODALITY}-responses.json"
    print(f"Loading {answers_conversations_file}")
    
    if not os.path.exists(answers_conversations_file):
        print(f'File does not exist! {answers_conversations_file}')
        continue

    answers = load_json(answers_conversations_file)
    answers = [a['content'] if 'content' in a else None for a in answers]
    
    clean_answers = [clean(a) for a in answers]

    OUTPUT_DIR, LAST_BATCH, all_responses = check_already_generated_evaluations(model_name)
    print(OUTPUT_DIR)
    SAVE_BATCH_SIZE = 100
    CURRENT_B = 0 if LAST_BATCH is None else LAST_BATCH + 1

    print(LAST_BATCH, len(all_responses))

    data['answer'] = clean_answers

    responses = []
    for i, row in tqdm(data.iterrows(), total=len(data)):
    
        if len(all_responses) != 0:
            if i < len(all_responses):
                continue

        if len(responses) % SAVE_BATCH_SIZE == 0 and len(responses) != 0:
            filename = f"{OUTPUT_DIR}/batch-{CURRENT_B}.json"
            print(f"Saving batch at {filename}")
            save_json(responses, filename)
            responses = []
            CURRENT_B+=1
        
        if row['answer'] is np.nan:
            print("Error of data at row", i)
            responses.append({'idx':i, 'error': 'Error in data'})
            continue

        secret = row['secret']
        memories = '-'+'\n-'.join(row['secret_holder']['personal_notes'])
        target_sentence = row['answer']

        try:
            response = wrapper.generate(secret, memories, target_sentence, i, verbose=i==0, ic=False)

            responses.append(response)
            if i < 10:
                print(response)
        except ClientError as e:
            print(e)
            #sleep(2)
            try:
                response = wrapper.generate(secret, memories, target_sentence, i, verbose=i==0, ic=False)
                responses.append(response)
                if i < 10:
                    print(response)
            except ClientError as e:
                print(e)
                #sleep(2)
                responses.append({'idx':i, 'error': str(e)})
                continue
        #sleep(2)
        
    if len(responses) > 0:
        filename = f"{OUTPUT_DIR}/batch-{CURRENT_B}.json"
        print(f"Saving batch at {filename}")
        save_json(responses, filename)    
        
    



defenses= ['CIMem_high', 'CIMem_medium', 'CIMem_low', "PrivacyChecker"]# ] # 

for DEFENSE_STRATEGY in defenses:
    print("-"*100)
    print(DEFENSE_STRATEGY)
    print("-"*100)
    for model_name in model_names:
        print("*"*80)
        print(model_name) 
        print("*"*80)
        answers_conversations_file = f"answers_conversations/{GROUP_TYPE}-social-{DEFENSE_STRATEGY}/{model_name}/test-conversation-ub{MODALITY}-responses.json"
        print(f"Loading {answers_conversations_file}")
        
        if not os.path.exists(answers_conversations_file):
            print(f'File does not exist! {answers_conversations_file}')
            continue

        answers = load_json(answers_conversations_file)
        answers = [a['content'] if 'content' in a else None for a in answers]

        clean_answers = [clean(a, defense=DEFENSE_STRATEGY if DEFENSE_STRATEGY == 'PrivacyChecker' else None) for a in answers]
        print(clean_answers[0:100])
        save_json(clean_answers, f'clean_answers_{model_name.split('/')[-1]}.json')
                
        OUTPUT_DIR, LAST_BATCH, all_responses = check_already_generated_evaluations(model_name, defense=DEFENSE_STRATEGY)
        
        SAVE_BATCH_SIZE = 100
        CURRENT_B = 0 if LAST_BATCH is None else LAST_BATCH + 1

        print(LAST_BATCH, len(all_responses))

        data['answer'] = clean_answers

        responses = []
        for i, row in tqdm(data.iterrows(), total=len(data)):
        
            if len(all_responses) != 0:
                if i < len(all_responses):
                    continue

            if len(responses) % SAVE_BATCH_SIZE == 0 and len(responses) != 0:
                filename = f"{OUTPUT_DIR}/batch-{CURRENT_B}.json"
                print(f"Saving batch at {filename}")
                save_json(responses, filename)
                responses = []
                CURRENT_B+=1
            
            if row['answer'] is np.nan:
                print("Error of data at row", i)
                responses.append({'idx':i, 'error': 'Error in data'})
                continue

            secret = row['secret']
            memories = '-'+'\n-'.join(row['secret_holder']['personal_notes'])
            target_sentence = row['answer']

            try:
                response = wrapper.generate(secret, memories, target_sentence, i, verbose=i==0, ic=False)

                responses.append(response)
                if i < 10:
                    print(response)
            except ClientError as e:
                print(e)
                #sleep(2)
                try:
                    response = wrapper.generate(secret, memories, target_sentence, i, verbose=i==0, ic=False)
                    responses.append(response)
                    if i < 10:
                        print(response)
                except ClientError as e:
                    print(e)
                    #sleep(2)
                    responses.append({'idx':i, 'error': str(e)})
                    continue
            #sleep(2)
            
        if len(responses) > 0:
            filename = f"{OUTPUT_DIR}/batch-{CURRENT_B}.json"
            print(f"Saving batch at {filename}")
            save_json(responses, filename)    
        
    
def clean_eval(a):
    if 'content' in a:
        if 'answer' in a['content']:
            return a['content']['answer']
        elif '"answer": "YES"' in a['content']['text-content']:
            #print(a['content']['text-content'])
            return 'YES'
        elif '"answer": "NO"' in a['content']['text-content']:
            #print(a['content']['text-content'])
            return 'NO'
    print(a)    
    return np.nan



In [ ]:
results = {}
for model_name in model_names:
    model_name_out = model_name.split('/')[1] if '/' in model_name else model_name
    print("*"*80)
    print(model_name)
    print("*"*80)

    OUTPUT_DIR, LAST_BATCH, all_responses = check_already_generated_evaluations(model_name)
    if len(all_responses) < len(data):
        print(f"Generation not complete for {model_name}")
        continue
    results[model_name_out] = [clean_eval(x) for x in all_responses] #['content']['answer'] if 'content' in x else np.nan
    print(len(results[model_name_out]))


for defense in defenses:
    for model_name in model_names:
        model_name_out = model_name.split('/')[1] if '/' in model_name else model_name
        print("*"*80)
        print(model_name)
        print("*"*80)

        OUTPUT_DIR, LAST_BATCH, all_responses = check_already_generated_evaluations(model_name, defense=defense)
        print(OUTPUT_DIR, f"{model_name_out}-{defense}")
        if len(all_responses) < len(data):
            print(f"Generation not complete for {model_name} - {defense}")
            continue
        results[f"{model_name_out}-{defense}"] = [clean_eval(x) for x in all_responses] #['content']['answer'] if 'content' in x else np.nan
        print(len(results[f"{model_name_out}-{defense}"]))


rows = []
for model in results:
    if model.startswith('human'):
        continue
    is_defended = False
    for defense in defenses:
        if defense in model:
            is_defended = True
    
    values = results[model]
    if not is_defended:
        model, conf = model, 'original'
    else:
        model, conf = '-'.join(model.split('-')[:-1]), model.split('-')[-1]
    #print(model, conf)
    for v in values:
        rows.append({"Model": model, "Conf":conf, "Response": v})
    
    

df = pd.DataFrame(rows)
# Count occurrences
counts = df.groupby(["Model", "Conf", "Response"]).size().reset_index(name="Count")
counts.Model = counts.Model.astype("category")
counts = counts.sort_values(by=['Model', 'Conf'])
# Compute percentages within each model
counts["Percentage"] = counts.groupby(["Model", "Conf"])["Count"].transform(lambda x: x / x.sum() * 100)
c = counts[(counts['Response']=='YES')].sort_values(by=['Model', "Conf"]).set_index(['Model', "Conf"])[['Count', 'Percentage']].round(2).T
c.to_csv('leakperc_table.csv')
pd.set_option('display.max_columns', None)
display(c)

In [ ]:
df = pd.DataFrame(rows)
# Count occurrences
counts = df.groupby(["Model", "Conf", "Response"]).size().reset_index(name="Count")
counts.Model = counts.Model.astype("category")
counts = counts.sort_values(by=['Model', 'Conf'])
counts = counts[counts["Conf"] == 'original']
# Compute percentages within each model
counts["Percentage"] = counts.groupby(["Model", "Conf"])["Count"].transform(lambda x: x / x.sum() * 100)
c = counts[(counts['Response']=='YES')].sort_values(by=['Model', "Conf"]).set_index(['Model', "Conf"])[['Count', 'Percentage']].round(2).T
c.to_csv('leakperc_table.csv')
pd.set_option('display.max_columns', None)
display(c)

In [ ]:
df = pd.DataFrame(rows)
# Count occurrences
counts = df.groupby(["Model", "Conf", "Response"]).size().reset_index(name="Count")
counts.Model = counts.Model.astype("category")
counts = counts.sort_values(by=['Model', 'Conf'])
counts = counts[counts["Conf"] != 'original']
# Compute percentages within each model
counts["Percentage"] = counts.groupby(["Model", "Conf"])["Count"].transform(lambda x: x / x.sum() * 100)
c = counts[(counts['Response']=='YES')].sort_values(by=['Model', "Conf"]).set_index(['Model', "Conf"])[['Count', 'Percentage']].round(2).T
c.to_csv('leakperc_table.csv')
pd.set_option('display.max_columns', None)
display(c)